# Notebook 04 — Fraud Strategy Design

**Project:** Fraud Detection & Strategy Analytics  
**Objective:** Translate model scores into an operational fraud strategy by defining score tiers, quantifying business tradeoffs, and recommending an optimal strategy configuration with a champion-challenger framework.

---

## The Fraud Strategy Problem

A model score alone does not make decisions — a strategy does. The strategy layer sits between the model output and the final customer-facing action, answering:

- **At what score do we auto-decline?** → Highest risk band
- **At what score do we queue for review?** → Medium risk band  
- **What is the cost of each false positive?** → Revenue / customer experience
- **What is the cost of each false negative?** → Fraud loss + chargeback cost

The art of fraud strategy is finding the **optimal tradeoff** between fraud prevention and customer friction.

---

## Outline
1. Load test scores
2. Score distribution analysis
3. Build tradeoff table
4. Visualise tradeoffs
5. Recommend optimal strategy
6. Champion-Challenger framework
7. Rule-based pre-filters
8. Full hybrid strategy demonstration

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from src.strategy import (
    ScoreCutoffs,
    FraudStrategyEngine,
    RuleBasedFilter,
    make_decision,
    DecisionTier,
)

# If the test scores CSV exists (from Notebook 03), load it;
# otherwise regenerate from scratch.
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
print('Imports complete.')

## 1. Load Scores

We load the XGBoost fraud probability scores generated in Notebook 03.  
If that notebook has not been run, we fall back to a quick simulation.

In [ ]:
scores_path = Path('../data/test_scores.csv')

if scores_path.exists():
    scores_df = pd.read_csv(scores_path)
    print(f'Loaded {len(scores_df):,} test scores from {scores_path}')
else:
    # Fallback: simulate realistic score distributions
    print('test_scores.csv not found — generating simulated scores.')
    rng = np.random.default_rng(42)
    n_legit = 19_600
    n_fraud = 400
    legit_scores = np.clip(rng.beta(1.5, 8, n_legit), 0, 1)
    fraud_scores = np.clip(rng.beta(5, 2,  n_fraud), 0, 1)
    scores_df = pd.DataFrame({
        'fraud_score': np.concatenate([legit_scores, fraud_scores]),
        'is_fraud':    np.concatenate([np.zeros(n_legit), np.ones(n_fraud)]),
        'transaction_amount': np.concatenate([
            rng.lognormal(4.0, 1.0, n_legit),
            rng.lognormal(5.0, 1.0, n_fraud),
        ]),
    })

y_true  = scores_df['is_fraud'].values.astype(int)
scores  = scores_df['fraud_score'].values
amounts = scores_df['transaction_amount'].values

print(f'Total: {len(scores_df):,} | Fraud: {y_true.sum():,} ({y_true.mean():.2%})')

## 2. Score Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE plot: score by class
for label, color, name in [(0, 'steelblue', 'Legitimate'), (1, 'tomato', 'Fraud')]:
    mask = y_true == label
    axes[0].hist(scores[mask], bins=50, alpha=0.6, color=color, label=name, density=True)

axes[0].set_xlabel('Fraud Probability Score')
axes[0].set_ylabel('Density')
axes[0].set_title('Score Distribution by Class')
axes[0].legend()

# Score percentile table
percentiles = [50, 75, 90, 95, 99]
fraud_pcts  = np.percentile(scores[y_true == 1], percentiles)
legit_pcts  = np.percentile(scores[y_true == 0], percentiles)

pct_df = pd.DataFrame({
    'Percentile': [f'P{p}' for p in percentiles],
    'Fraud Score': fraud_pcts.round(3),
    'Legit Score': legit_pcts.round(3),
})

axes[1].axis('off')
table = axes[1].table(
    cellText=pct_df.values,
    colLabels=pct_df.columns,
    cellLoc='center',
    loc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)
axes[1].set_title('Score Percentiles by Class', pad=40)

plt.tight_layout()
plt.show()

## 3. Build Tradeoff Table

For each candidate **decline threshold**, we compute:  
- **Fraud catch rate**: What % of fraud do we auto-decline?  
- **False positive rate**: What % of legitimate transactions do we block?  
- **Review volume**: How many cases hit the analyst queue?  
- **Net benefit**: Revenue impact of preventing fraud minus cost of blocking legitimate spend and running the review queue.

In [ ]:
engine = FraudStrategyEngine(
    fraud_scores=scores,
    y_true=y_true,
    amounts=amounts,
    review_offset=0.2,
)

tradeoff_table = engine.build_tradeoff_table(
    decline_cutoffs=[0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
)

print('=== STRATEGY TRADEOFF TABLE ===')
print(tradeoff_table.to_string(index=False))

## 4. Visualise Tradeoffs

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

cutoffs_x = tradeoff_table['decline_cutoff']

# --- Top left: Fraud catch rate vs False positive rate ---
ax = axes[0, 0]
ax.plot(cutoffs_x, tradeoff_table['fraud_catch_rate_%'],
        'o-', color='tomato', label='Fraud catch rate')
ax.plot(cutoffs_x, tradeoff_table['false_positive_rate_%'],
        's-', color='steelblue', label='False positive rate')
ax.set_xlabel('Decline Threshold')
ax.set_ylabel('%')
ax.set_title('Fraud Catch Rate vs False Positive Rate')
ax.legend()

# --- Top right: Net benefit ---
ax = axes[0, 1]
colors = ['tomato' if v < 0 else 'seagreen' for v in tradeoff_table['net_benefit_usd']]
ax.bar(cutoffs_x, tradeoff_table['net_benefit_usd'] / 1000, color=colors, edgecolor='white')
ax.set_xlabel('Decline Threshold')
ax.set_ylabel('Net Benefit ($000s)')
ax.set_title('Net Revenue Benefit by Decline Threshold')
ax.axhline(0, color='black', linewidth=0.8)

# --- Bottom left: Review volume ---
ax = axes[1, 0]
ax2 = ax.twinx()
ax.bar(cutoffs_x, tradeoff_table['review_volume'], color='mediumpurple', alpha=0.7, label='Review volume')
ax2.plot(cutoffs_x, tradeoff_table['ops_cost_usd'] / 1000, 'D-', color='darkorange', label='Ops cost ($000s)')
ax.set_xlabel('Decline Threshold')
ax.set_ylabel('Cases in Review Queue')
ax2.set_ylabel('Ops Cost ($000s)')
ax.set_title('Review Volume & Operations Cost')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

# --- Bottom right: Total fraud stopped ---
ax = axes[1, 1]
ax.plot(cutoffs_x, tradeoff_table['total_fraud_caught_%'],
        'P-', color='seagreen', lw=2, markersize=8)
ax.fill_between(cutoffs_x, tradeoff_table['total_fraud_caught_%'], alpha=0.15, color='seagreen')
ax.set_xlabel('Decline Threshold')
ax.set_ylabel('Total Fraud Stopped (%)')
ax.set_title('Total Fraud Stopped (Auto-decline + Review)')
ax.set_ylim(0, 110)

plt.suptitle('Fraud Strategy Tradeoff Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Recommended Strategy

In [ ]:
recommendation = engine.recommend(
    objective='net_benefit',
    min_fraud_catch_rate=0.50,     # Must catch at least 50% of fraud
    max_false_positive_rate=0.05,  # Must not block > 5% of legitimate txns
)

print(recommendation.rationale)

In [ ]:
# Visualise score tiers for recommended strategy
cutoffs = recommendation.cutoffs

fig, ax = plt.subplots(figsize=(12, 5))

for label, color, name in [(0, 'steelblue', 'Legitimate'), (1, 'tomato', 'Fraud')]:
    mask = y_true == label
    ax.hist(scores[mask], bins=80, alpha=0.5, color=color, density=True, label=name)

# Shade decision zones
ax.axvspan(0, cutoffs.review_threshold,  alpha=0.08, color='seagreen', label='APPROVE zone')
ax.axvspan(cutoffs.review_threshold, cutoffs.decline_threshold, alpha=0.08, color='gold', label='REVIEW zone')
ax.axvspan(cutoffs.decline_threshold, 1, alpha=0.08, color='tomato', label='DECLINE zone')

ax.axvline(cutoffs.review_threshold,  color='darkorange', linestyle='--', lw=2,
           label=f'Review cutoff ({cutoffs.review_threshold:.2f})')
ax.axvline(cutoffs.decline_threshold, color='darkred',    linestyle='--', lw=2,
           label=f'Decline cutoff ({cutoffs.decline_threshold:.2f})')

ax.set_xlabel('Fraud Probability Score')
ax.set_ylabel('Density')
ax.set_title('Recommended Strategy: Score Tiers')
ax.legend(loc='upper center', ncol=3, fontsize=9)

# Annotate zones
ax.text(cutoffs.review_threshold / 2, ax.get_ylim()[1] * 0.85,
        'APPROVE', ha='center', color='seagreen', fontsize=11, fontweight='bold')
ax.text((cutoffs.review_threshold + cutoffs.decline_threshold) / 2, ax.get_ylim()[1] * 0.85,
        'REVIEW', ha='center', color='darkorange', fontsize=11, fontweight='bold')
ax.text((cutoffs.decline_threshold + 1) / 2, ax.get_ylim()[1] * 0.85,
        'DECLINE', ha='center', color='darkred', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Champion-Challenger Framework

In production fraud strategy, we never replace the current strategy ('champion') directly. Instead we run a **champion-challenger test** where a small percentage of traffic is routed to the new strategy ('challenger') and results are compared statistically before any promotion.

This controls for population drift, seasonal effects, and avoids irreversible decisions.

In [ ]:
cc_config = recommendation.champion_challenger

print('=== CHAMPION-CHALLENGER CONFIGURATION ===')
for role, config in [('CHAMPION', cc_config['champion']), ('CHALLENGER', cc_config['challenger'])]:
    print(f'\n  [{role}]')
    for k, v in config.items():
        print(f'    {k:25s}: {v}')

print(f'\n  Success metric      : {cc_config["success_metric"]}')
print(f'  Min sample size     : {cc_config["minimum_sample_size"]:,}')
print(f'  Evaluation cadence  : {cc_config["evaluation_cadence"]}')
print(f'\n  Notes: {cc_config["notes"]}')

In [ ]:
# Simulate traffic split
rng = np.random.default_rng(42)
traffic_assignment = rng.choice(
    ['champion', 'challenger'],
    size=len(scores),
    p=[cc_config['champion']['traffic_pct'] / 100,
       cc_config['challenger']['traffic_pct'] / 100]
)

# Apply each strategy to its traffic
champ_cutoffs  = ScoreCutoffs(
    review_threshold=cc_config['champion']['review_threshold'],
    decline_threshold=cc_config['champion']['decline_threshold'],
)
chall_cutoffs  = ScoreCutoffs(
    review_threshold=cc_config['challenger']['review_threshold'],
    decline_threshold=cc_config['challenger']['decline_threshold'],
)

for label, cutoff_obj in [('Champion', champ_cutoffs), ('Challenger', chall_cutoffs)]:
    mask = traffic_assignment == label.lower()
    sub_scores = scores[mask]
    sub_y = y_true[mask]
    decisions = cutoff_obj.assign(sub_scores)
    n_declined = (decisions == DecisionTier.DECLINE).sum()
    n_fraud_declined = ((decisions == DecisionTier.DECLINE) & (sub_y == 1)).sum()
    catch_rate = n_fraud_declined / max(sub_y.sum(), 1)
    print(f'{label}: n={mask.sum():,} | fraud_declined={n_fraud_declined} | catch_rate={catch_rate:.1%}')

## 7. Rule-Based Pre-Filters

Hard rules provide immediate, deterministic declines for well-understood, high-confidence fraud patterns. They are:
- Faster than model scoring (no latency)
- Easier to audit for compliance
- Applied **before** the ML score to reduce false negatives for known attack patterns

In [ ]:
from data.generate_data import generate_dataset
from src.data_processing import clean_data, split_data

# Reload raw data for rule testing
df_raw = generate_dataset(n=100_000)
df_clean = clean_data(df_raw)

df_with_rules = RuleBasedFilter.apply(df_clean)

rule_summary = (
    df_with_rules.groupby('rule_triggered')
    .agg(
        transactions=('transaction_id', 'count'),
        fraud_caught=('is_fraud', 'sum'),
    )
    .assign(precision=lambda x: x['fraud_caught'] / x['transactions'])
    .sort_values('transactions', ascending=False)
)

print('Rule-based filter performance:')
print(rule_summary.to_string())

> **Interpretation:**
> - Rules flagging `R2_location_prior_fraud` should have the highest precision (both signals together are highly reliable).
> - `R1_velocity` may have more false positives but catches a high volume of fraud.
> - Rules should be reviewed quarterly — fraudsters adapt.

## 8. Full Hybrid Strategy Demonstration

In [ ]:
# Sample 1,000 transactions for demonstration
sample = df_clean.sample(1000, random_state=42).reset_index(drop=True)

# Simulate fraud scores (use actual scores if available)
rng_demo = np.random.default_rng(99)
demo_scores = np.where(
    sample['is_fraud'] == 1,
    np.clip(rng_demo.beta(5, 2,  len(sample)), 0, 1),
    np.clip(rng_demo.beta(1, 8,  len(sample)), 0, 1),
)

result = make_decision(sample, demo_scores, recommendation.cutoffs)

# Decision distribution
decision_counts = result['decision'].value_counts()
print('Decision distribution:')
print(decision_counts.to_string())

# Fraud catch by tier
print('\nFraud captured per decision tier:')
print(result.groupby('decision')['is_fraud'].agg(['sum', 'mean']).rename(
    columns={'sum': 'fraud_count', 'mean': 'fraud_rate'}).to_string())

In [ ]:
# Decision pie chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colours = {
    DecisionTier.APPROVE:  'seagreen',
    DecisionTier.REVIEW:   'gold',
    DecisionTier.DECLINE:  'tomato',
}

# Overall decision split
tier_order = [DecisionTier.APPROVE, DecisionTier.REVIEW, DecisionTier.DECLINE]
tier_counts = [decision_counts.get(t, 0) for t in tier_order]
tier_colors = [colours[t] for t in tier_order]

axes[0].pie(tier_counts, labels=tier_order, colors=tier_colors,
            autopct='%1.1f%%', startangle=90)
axes[0].set_title('Transaction Decision Split')

# Fraud concentration by tier
tier_fraud = result.groupby('decision')['is_fraud'].sum().reindex(tier_order, fill_value=0)
axes[1].bar(tier_order, tier_fraud.values, color=tier_colors, edgecolor='white')
axes[1].set_title('Fraud Transactions by Decision Tier')
axes[1].set_ylabel('Number of Fraud Cases')
for i, v in enumerate(tier_fraud.values):
    axes[1].text(i, v + 0.3, str(v), ha='center', fontsize=12)

plt.tight_layout()
plt.show()

## 9. Strategy Summary

### Final Recommended Configuration

| Parameter | Value |
|-----------|-------|
| APPROVE threshold | Score < REVIEW cutoff |
| REVIEW threshold  | Score ≥ review_threshold |
| DECLINE threshold | Score ≥ decline_threshold |
| Rule pre-filters  | R1 (velocity), R2 (location + prior fraud), R3 (new acct + high amt) |

### Key Takeaways

1. **No single threshold is optimal** — a three-tier strategy (approve/review/decline) outperforms a binary cutoff by directing analyst attention only where it adds value.

2. **Rules + ML is better than either alone** — hard rules catch known patterns with zero latency; ML generalises to novel fraud patterns.

3. **Tradeoff is an explicit business decision** — the recommended cutoffs balance fraud prevention against false positive cost. Different risk appetites require different configurations.

4. **Champion-Challenger testing is mandatory** before any strategy change to measure real-world impact under controlled conditions.

5. **Model refresh cadence** should be quarterly (or triggered by KS drift > 10% from baseline) to maintain performance as fraud patterns evolve.